# 🛡️ Glu-Stock: 00a_MODEL_RETRAINING_RF
**Phase**: Dynamic Cross-Sectional Intelligence (RF Brain)

This notebook trains the Random Forest 'Super Brain' on a dynamic panel dataset containing 5 years of historical data from the latest active LQ45 constituents. It scrapes the current LQ45 members to ensure the model stays relevant to the most liquid assets.

In [11]:
!pip install -q yfinance firebase-admin pandas scikit-learn joblib python-dotenv ta optuna lightgbm imbalanced-learn


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Web Fetchers)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf
from firebase_admin import credentials, firestore
from datetime import datetime

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try: tg = user_secrets.get_secret("TELEGRAM_TOKEN")
            except: tg = None
            return {
                "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON")),
                "telegram": tg
            }
        else:
            from dotenv import load_dotenv
            load_dotenv()
            return {
                "key": json.loads(os.getenv("FIREBASE_KEY_JSON", "{}")),
                "telegram": os.getenv("TELEGRAM_TOKEN")
            }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()
        
    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
        
    def push_task(self, queue_name: str, data):
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})
        
    def insert_trade(self, trade_data):
        self.db.collection("glu_stock_trades").add(trade_data)
        
    def get_history(self, limit=5):
        docs = self.db.collection("glu_stock_history").order_by("timestamp", direction=firestore.Query.DESCENDING).limit(limit).get()
        history = [doc.to_dict() for doc in docs]
        return {str(i): h for i, h in enumerate(reversed(history))} if history else {}
        
    def get_active_trades(self):
        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()
        return [doc.to_dict() for doc in docs]
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

def get_dynamic_lq45():
    print("🌐 Fetching latest LQ45 constituents...")
    fallback = ["ACES.JK", "ADRO.JK", "AKRA.JK", "AMMN.JK", "AMRT.JK", "ANTM.JK", "ARTO.JK", "ASII.JK", "BBCA.JK", "BBNI.JK", "BBRI.JK", "BBTN.JK", "BMRI.JK", "BRIS.JK", "BRPT.JK", "BUKA.JK", "CPIN.JK", "CTRA.JK", "ESSA.JK", "EXCL.JK", "GGRM.JK", "GOTO.JK", "HRUM.JK", "ICBP.JK", "INCO.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK", "ITMG.JK", "KLBF.JK", "MAPI.JK", "MBMA.JK", "MDKA.JK", "MEDC.JK", "MTEL.JK", "PGAS.JK", "PGEO.JK", "PTBA.JK", "SIDO.JK", "SMGR.JK", "SRTG.JK", "TLKM.JK", "TPIA.JK", "UNTR.JK"]
    try:
        import urllib.request
        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/lq45.json', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=5) as url:
            data = json.loads(url.read().decode())
            return [f"{t}.JK" for t in data]
    except Exception as e:
        print(f"⚠️ Github LQ45 fetch failed: {e}. Using highly-curated fallback LQ45 list.")
    return fallback

def get_full_idx_universe():
    print("🌐 Fetching ALL IDX listed companies from Official IDX API...")
    fallback = ['AALI.JK', 'ABMM.JK', 'ACES.JK', 'ADHI.JK', 'AISA.JK', 'AKRA.JK', 'AMRT.JK', 'ANTM.JK', 'APLN.JK', 'ARNA.JK', 'ARTO.JK', 'ASGR.JK', 'ASII.JK', 'ASRI.JK', 'ASSA.JK', 'AUTO.JK', 'BACA.JK', 'BALI.JK', 'BAYU.JK', 'BBCA.JK', 'BBHI.JK', 'BBNI.JK', 'BBRI.JK', 'BBTN.JK', 'BBYB.JK', 'BCAP.JK', 'BDMN.JK', 'BEST.JK', 'BFIN.JK', 'BGTG.JK', 'BINA.JK', 'BIRD.JK', 'BISI.JK', 'BJBR.JK', 'BJTM.JK', 'BKSL.JK', 'BMRI.JK', 'BMTR.JK', 'BNGA.JK', 'BNII.JK', 'BNLI.JK', 'BRMS.JK', 'BRPT.JK', 'BSDE.JK', 'BSIM.JK', 'BTPN.JK', 'BUDI.JK', 'BUKK.JK', 'BUMI.JK', 'BVIC.JK', 'BWPT.JK', 'BYAN.JK', 'CASS.JK', 'CFIN.JK', 'CITA.JK', 'CMNP.JK', 'CPIN.JK', 'CTRA.JK', 'DEWA.JK', 'DILD.JK', 'DLTA.JK', 'DMAS.JK', 'DNET.JK', 'DOID.JK', 'DSNG.JK', 'DSSA.JK', 'ELSA.JK', 'EMTK.JK', 'ENRG.JK', 'ERAA.JK', 'ESSA.JK', 'EXCL.JK', 'GEMS.JK', 'GGRM.JK', 'GJTL.JK', 'GWSA.JK', 'HEXA.JK', 'HMSP.JK', 'HRUM.JK', 'ICBP.JK', 'IMAS.JK', 'IMPC.JK', 'INCO.JK', 'INDF.JK', 'INDY.JK', 'INKP.JK', 'INPC.JK', 'INTP.JK', 'ISAT.JK', 'ISSP.JK', 'ITMG.JK', 'JKON.JK', 'JPFA.JK', 'JRPT.JK', 'JSMR.JK', 'JTPE.JK', 'KBLI.JK', 'KIJA.JK', 'KKGI.JK', 'KLBF.JK', 'KPIG.JK', 'LPKR.JK', 'LPPF.JK', 'LSIP.JK', 'LTLS.JK', 'MAIN.JK', 'MAPI.JK', 'MAYA.JK', 'MBSS.JK', 'MCOR.JK', 'MDKA.JK', 'MEDC.JK', 'MEGA.JK', 'MIDI.JK', 'MIKA.JK', 'MLBI.JK', 'MLIA.JK', 'MLPL.JK', 'MMLP.JK', 'MNCN.JK', 'MPMX.JK', 'MREI.JK', 'MTDL.JK', 'MTLA.JK', 'MYOR.JK', 'NISP.JK', 'PANR.JK', 'PANS.JK', 'PGAS.JK', 'PNBN.JK', 'PNIN.JK', 'PNLF.JK', 'PTBA.JK', 'PTPP.JK', 'PTRO.JK', 'PWON.JK', 'RAJA.JK', 'RALS.JK', 'SAME.JK', 'SCMA.JK', 'SGRO.JK', 'SIDO.JK', 'SILO.JK', 'SIMP.JK', 'SMAR.JK', 'SMBR.JK', 'SMDR.JK', 'SMGR.JK', 'SMMA.JK', 'SMRA.JK', 'SMSM.JK', 'SRTG.JK', 'SSIA.JK', 'SSMS.JK', 'TBIG.JK', 'TBLA.JK', 'TINS.JK', 'TKIM.JK', 'TLKM.JK', 'TMAS.JK', 'TOBA.JK', 'TOTL.JK', 'TOWR.JK', 'TPMA.JK', 'TRIM.JK', 'TSPC.JK', 'ULTJ.JK', 'UNIC.JK', 'UNTR.JK', 'UNVR.JK', 'VICO.JK', 'WIIM.JK', 'WINS.JK', 'WTON.JK', 'SHIP.JK', 'POWR.JK', 'PRDA.JK', 'BRIS.JK', 'CARS.JK', 'CLEO.JK', 'WOOD.JK', 'HRTA.JK', 'MARK.JK', 'MCAS.JK', 'PSSI.JK', 'MORA.JK', 'PBID.JK', 'IPCM.JK', 'BTPS.JK', 'SPTO.JK', 'HEAL.JK', 'TUGU.JK', 'MSIN.JK', 'MAPA.JK', 'IPCC.JK', 'FILM.JK', 'PANI.JK', 'GOOD.JK', 'SKRN.JK', 'BOLA.JK', 'KOTA.JK', 'KEEN.JK', 'TEBE.JK', 'KEJU.JK', 'PSGO.JK', 'UCID.JK', 'CSRA.JK', 'SAMF.JK', 'SGER.JK', 'PNGO.JK', 'BBSI.JK', 'VICI.JK', 'WMUU.JK', 'UNIQ.JK', 'TAPG.JK', 'BMHS.JK', 'MCOL.JK', 'GTSI.JK', 'MTEL.JK', 'CMRY.JK', 'RMKE.JK', 'AVIA.JK', 'DRMA.JK', 'ADMR.JK', 'STAA.JK', 'MTMH.JK', 'TRGU.JK', 'HATM.JK', 'JARR.JK', 'ELPI.JK', 'MKTR.JK', 'OMED.JK', 'SUNI.JK', 'PGEO.JK', 'BDKR.JK', 'CUAN.JK', 'SMIL.JK', 'AMMN.JK', 'MAHA.JK', 'ERAL.JK', 'BREN.JK', 'MSTI.JK', 'ALII.JK', 'GOLF.JK', 'DAAZ.JK', 'AADI.JK', 'MDIY.JK', 'DGWG.JK', 'CBDK.JK', 'MINE.JK', 'PSAT.JK', 'BLOG.JK', 'YUPI.JK', 'MDLA.JK', 'NCKL.JK', 'MBMA.JK', 'RAAM.JK', 'ADRO.JK', 'AGRO.JK']
    
    # 1. Try Official IDX API
    try:
        import urllib.request
        hdrs = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Referer': 'https://www.idx.co.id/'
        }
        req = urllib.request.Request('https://www.idx.co.id/primary/StockData/GetSecuritiesStock?length=9999', headers=hdrs)
        with urllib.request.urlopen(req, timeout=10) as url:
            data = json.loads(url.read().decode())
            if 'data' in data:
                tickers = [f"{t['Code']}.JK" for t in data['data'] if 'Code' in t]
                if tickers:
                    print(f"✅ Successfully fetched {len(tickers)} companies from IDX Official API.")
                    return list(set(tickers)) 
    except Exception as e:
        print(f"⚠️ Official IDX API failed: {e}. Trying Github Proxy...")
        
    # 2. Try Github Alternative
    try:
        import urllib.request
        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/stock-list.json', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as url:
            data = json.loads(url.read().decode())
            tickers = [f"{t['ticker']}.JK" for t in data if 'ticker' in t]
            if tickers:
                print(f"✅ Successfully fetched {len(tickers)} companies from Github proxy.")
                return list(set(tickers))
    except Exception as e:
        print(f"⚠️ Full fetch failed: {e}. Falling back to MASTER 259 Papan Utama list.")
        
    return fallback
\n

SyntaxError: unexpected character after line continuation character (1535006911.py, line 2)

In [ ]:
# 🧠 SECTION 3: CORE LOGIC (LightGBM Alpha Pipeline)
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import mutual_info_classif
from sklearn.base import clone
import numpy as np, pandas as pd, ta, optuna, warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

def frac_diff(series, d=0.4, window=100):
    w = [1.0]
    for k in range(1, window):
        w.append(-w[-1] * (d - k + 1) / k)
    w = np.array(w[::-1])
    result = np.full(len(series), np.nan)
    for t in range(window - 1, len(series)):
        result[t] = np.dot(w, series[t - window + 1:t + 1])
    return result

def triple_barrier_label_2class(close_arr, t_horizon=10, sl_pct=0.02, tp_pct=0.03):
    labels = np.zeros(len(close_arr))
    for i in range(len(close_arr) - t_horizon):
        future = close_arr[i+1:i+1+t_horizon]
        entry = close_arr[i]
        if entry == 0:
            continue
        tp_hit = np.any(future >= entry * (1 + tp_pct))
        sl_hit = np.any(future <= entry * (1 - sl_pct))
        tp_idx = np.argmax(future >= entry * (1 + tp_pct)) if tp_hit else t_horizon
        sl_idx = np.argmax(future <= entry * (1 - sl_pct)) if sl_hit else t_horizon
        if tp_hit and (tp_idx <= sl_idx):
            labels[i] = 1
    return labels

def walk_forward_cv(model, X, y, feature_names, n_splits=4):
    chunk = len(X) // (n_splits + 1)
    scores = []
    for i in range(n_splits):
        train_end = chunk * (i + 2)
        test_end = min(train_end + chunk, len(X))
        if test_end <= train_end:
            continue
        m = clone(model)
        df_tr = pd.DataFrame(X[:train_end], columns=feature_names)
        df_te = pd.DataFrame(X[train_end:test_end], columns=feature_names)
        m.fit(df_tr, y[:train_end])
        scores.append(m.score(df_te, y[train_end:test_end]))
    return scores

def build_panel_data(universe, period='5y'):
    all_X, all_y = [], []
    feature_names = ['Returns','RSI','MACD','BB_High','BB_Low','ATR','ADX','OBV_norm','day_of_week','week_of_month','frac_diff_close','vol_ratio']
    n_total = len(universe)
    print(f'\U0001f4c9 Fetching {period} of data for {n_total} tickers...')
    skip_empty, skip_short, skip_dropna, fail_err, ok_count = 0, 0, 0, 0, 0
    for idx, ticker in enumerate(universe):
        if idx % 50 == 0:
            print(f'  ... processing {idx}/{n_total} ...')
        try:
            df = yf.download(ticker, period=period, progress=False, auto_adjust=True)
            if df is None or len(df) == 0:
                skip_empty += 1
                continue
            if len(df) < 150:
                skip_short += 1
                continue
            close = df['Close'].squeeze()
            high = df['High'].squeeze()
            low = df['Low'].squeeze()
            volume = df['Volume'].squeeze()
            if hasattr(close, 'ndim') and close.ndim > 1:
                close = close.iloc[:, 0]
            if hasattr(high, 'ndim') and high.ndim > 1:
                high = high.iloc[:, 0]
            if hasattr(low, 'ndim') and low.ndim > 1:
                low = low.iloc[:, 0]
            if hasattr(volume, 'ndim') and volume.ndim > 1:
                volume = volume.iloc[:, 0]
            feat = pd.DataFrame(index=df.index)
            feat['Returns'] = close.pct_change()
            feat['RSI'] = ta.momentum.RSIIndicator(close=close, window=14).rsi()
            macd_ind = ta.trend.MACD(close=close)
            feat['MACD'] = macd_ind.macd_diff()
            boll = ta.volatility.BollingerBands(close=close, window=20, window_dev=2)
            feat['BB_High'] = boll.bollinger_hband_indicator()
            feat['BB_Low'] = boll.bollinger_lband_indicator()
            feat['ATR'] = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()
            feat['ADX'] = ta.trend.ADXIndicator(high=high, low=low, close=close, window=14).adx()
            obv = ta.volume.OnBalanceVolumeIndicator(close=close, volume=volume).on_balance_volume()
            feat['OBV_norm'] = (obv - obv.rolling(20).mean()) / (obv.rolling(20).std() + 1e-7)
            feat['day_of_week'] = df.index.dayofweek
            feat['week_of_month'] = (df.index.day - 1) // 7
            feat['frac_diff_close'] = frac_diff(close.values.flatten(), d=0.4, window=100)
            vol_ma = volume.rolling(20).mean()
            feat['vol_ratio'] = volume / (vol_ma + 1e-7)
            feat = feat.dropna()
            if len(feat) < 50:
                skip_dropna += 1
                continue
            close_arr = close.reindex(feat.index).values.flatten()
            y = triple_barrier_label_2class(close_arr, t_horizon=10, sl_pct=0.02, tp_pct=0.03)
            valid = min(len(feat), len(y))
            X = feat[feature_names].values[:valid]
            y = y[:valid]
            mask = ~np.isnan(X).any(axis=1)
            if mask.sum() > 0:
                all_X.append(X[mask])
                all_y.append(y[mask])
                ok_count += 1
        except Exception as e:
            fail_err += 1
            if fail_err <= 5:
                print(f'\u26a0\ufe0f {ticker}: {type(e).__name__}: {e}')
            continue
    print(f'\U0001f4ca Pipeline: OK={ok_count} | Empty={skip_empty} | Short={skip_short} | Dropna={skip_dropna} | Error={fail_err}')
    if len(all_X) == 0:
        return None, None, feature_names
    Xr = np.vstack(all_X)
    yr = np.concatenate(all_y)
    print(f'\u2705 Aggregated {ok_count}/{n_total} tickers | {len(yr)} samples.')
    return Xr, yr, feature_names

def select_features(X, y, feature_names, top_k=10):
    print(f'\U0001f50d Feature Selection via Mutual Information...')
    mi = mutual_info_classif(X, y, random_state=42)
    ranked = sorted(zip(feature_names, mi), key=lambda x: -x[1])
    for name, score in ranked:
        print(f'  {name}: {score:.4f}')
    selected = [name for name, score in ranked[:top_k]]
    idx = [feature_names.index(s) for s in selected]
    print(f'\u2705 Selected top {top_k}: {selected}')
    return X[:, idx], selected

def optimize_lgbm(X, y, feature_names, n_trials=50):
    print(f'\U0001f50d Running Optuna HPO ({n_trials} trials)...')
    def objective(trial):
        model = LGBMClassifier(
            n_estimators=trial.suggest_int('n_estimators', 100, 500),
            max_depth=trial.suggest_int('max_depth', 4, 12),
            learning_rate=trial.suggest_float('lr', 0.01, 0.1, log=True),
            num_leaves=trial.suggest_int('num_leaves', 15, 63),
            min_child_samples=trial.suggest_int('min_child_samples', 5, 30),
            is_unbalance=True, verbose=-1, n_jobs=-1, random_state=42
        )
        scores = walk_forward_cv(model, X, y, feature_names, n_splits=4)
        return np.mean(scores) if scores else 0.0
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, timeout=600)
    print(f'\u2705 Optuna Best Val Acc: {study.best_value:.4f}')
    print(f'\u2705 Best Params: {study.best_params}')
    return study.best_params, study.best_value


In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_retrain():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    output_dir = '/kaggle/working/'
    
    universe = get_full_idx_universe()
    
    X, y, feature_names = build_panel_data(universe)
    if X is None:
        print('\u26a0\ufe0f Retraining aborted.')
        return
    
    print(f'\U0001f4ca Label Distribution: BUY={int(np.sum(y==1))}, DONT_BUY={int(np.sum(y==0))}')
    
    X, selected_features = select_features(X, y, feature_names, top_k=10)
    
    split_idx = int(len(X) * 0.85)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    print('\U0001f4a1 Applying SMOTE oversampling...')
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    print(f'\u2705 After SMOTE: {len(y_train_res)} samples (BUY={int(np.sum(y_train_res==1))}, DONT_BUY={int(np.sum(y_train_res==0))})')
    
    best_params, best_cv = optimize_lgbm(X_train_res, y_train_res, selected_features, n_trials=50)
    
    print('\U0001f3c6 Training Final LightGBM...')
    model = LGBMClassifier(
        n_estimators=best_params['n_estimators'],
        max_depth=best_params['max_depth'],
        learning_rate=best_params['lr'],
        num_leaves=best_params['num_leaves'],
        min_child_samples=best_params['min_child_samples'],
        is_unbalance=True, verbose=-1, n_jobs=-1, random_state=42
    )
    df_train = pd.DataFrame(X_train_res, columns=selected_features)
    df_test = pd.DataFrame(X_test, columns=selected_features)
    model.fit(df_train, y_train_res)
    train_acc = model.score(df_train, y_train_res)
    val_acc = model.score(df_test, y_test)
    
    oos_preds = model.predict(df_test)
    oos_correct = (oos_preds == y_test).astype(int).tolist()
    import json as _json
    with open(os.path.join(output_dir, 'rf_oos_meta.json'), 'w') as mf:
        _json.dump({'oos_correct': oos_correct, 'oos_preds': oos_preds.tolist(), 'val_acc': val_acc}, mf)
    
    brain_data = {
        'model': model,
        'features': selected_features,
        'best_params': best_params,
        'train_accuracy': train_acc,
        'val_accuracy': val_acc,
        'optuna_cv_score': best_cv,
        'model_type': 'LightGBM',
        'trained_at': datetime.now().isoformat()
    }
    joblib.dump(brain_data, os.path.join(output_dir, 'glu_brain_v1.joblib'))
    
    log_lines = [
        'LightGBM Training Complete.',
        f'\U0001f393 Training Acc: {train_acc:.2%}',
        f'\U0001f6e1\ufe0f OOS Validation Acc: {val_acc:.2%}',
        f'\U0001f50d Optuna Best CV: {best_cv:.2%}',
        f'\U0001f4ca Universe: {len(universe)} symbols | Samples: {len(y)}',
        f'\U0001f3af Labels: BUY={int(np.sum(y==1))}, DONT={int(np.sum(y==0))}',
        f'\U0001f9e0 Features: {len(selected_features)} | SMOTE: {len(y_train_res)}',
        f'\u2699\ufe0f Params: {best_params}',
    ]
    fb.log_event('RETRAINING_LGBM', chr(10).join(log_lines))
    for line in log_lines:
        print(line)

run_retrain()


🌐 Fetching latest LQ45 constituents...
📉 Fetching 5y of data for 45 tickers...



1 Failed download:
['ACES.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ADRO.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['AKRA.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['AMMN.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['AMRT.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ANTM.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ARTO.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['ASII.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBCA.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBNI.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBRI.JK']: TypeError("'NoneType' object is not subscriptable")

1 Failed download:
['BBTN.JK']: TypeError("'NoneType' object is 

✅ Successfully aggregated market data.


ValueError: need at least one array to concatenate